# Track B — Relation Classifier Data Pipeline

Notebook chạy trên **Google Colab Pro (A100 40GB)**. Bám theo [`PHASE2_PLAN.md`](PHASE2_PLAN.md).

**Kiểm soát tiến độ bằng [`PHASE2_CHECKLIST.md`](PHASE2_CHECKLIST.md)** — file đó là nguồn sự thật về việc bước nào đã xong, ngưỡng nào phải đạt, khi nào phải dừng.

---

### Nguyên tắc vận hành của notebook này

1. **Mọi stage đều resumable.** Output ghi xuống Drive; chạy lại cell đã xong sẽ **skip**, không tính tiền lại. Colab ngắt kết nối giữa chừng không mất việc.
2. **Ngân sách được theo dõi tự động.** Mỗi stage ghi thời gian GPU vào `budget_log.json`. Xem tổng bằng cell "Budget report".
3. **Model chạy tuần tự.** Nạp 1 model → chạy hết tập → giải phóng VRAM → nạp model kế. Không bao giờ 2 model cùng lúc.
4. **`PILOT_MODE = True` là mặc định.** Chạy pilot trước, trả lời 5 câu hỏi ở checklist, rồi mới đặt `False`.

### Thứ tự chạy

```
Setup → B1 → B2 → B2-guard → B3 → [PILOT GATE] → B4 → B5 → Debate → B6 → Spot-check → Dịch → Manifest
```

---
# 0 · Setup

## 0.1 · Cài đặt

Chạy một lần mỗi session. Colab sẽ yêu cầu restart runtime sau khi cài vLLM — restart rồi chạy lại từ cell 0.2 (không cần chạy lại 0.1).

In [ ]:
# Cài đặt. Sau cell này Colab có thể yêu cầu restart runtime -> restart rồi chạy tiếp từ 0.2
!pip install -q vllm==0.19.1
!pip install -q rapidfuzz ftfy transformers accelerate sentencepiece
print("Done. Nếu Colab báo restart runtime -> restart, rồi chạy từ cell 0.2 (bỏ qua 0.1).")

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Không thấy GPU. Runtime > Change runtime type > A100."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  |  VRAM: {vram:.1f} GB")
if "A100" not in name:
    print("\n[CẢNH BÁO] Không phải A100. Mọi ước lượng ngân sách trong plan tính theo A100 40GB (11.8 units/h).")
    print("           Throughput và chi phí units sẽ khác -> đo lại bằng pilot trước khi cam kết.")

## 0.2 · Mount Drive + cấu hình

**Vì sao bắt buộc Drive:** Colab ngắt kết nối là mất sạch `/content`. Pipeline này chạy nhiều giờ; mất giữa chừng mà không có checkpoint nghĩa là đốt units chạy lại từ đầu — mà ngân sách không có API dự phòng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, hashlib, random, gc, re
from pathlib import Path


# --- Xac thuc Hugging Face (bat buoc cho model gated nhu gemma-2-9b-it) ---
from huggingface_hub import login as _hf_login
try:
    from google.colab import userdata
    _hf_tok = userdata.get("HF_TOKEN")
except Exception:
    _hf_tok = os.environ.get("HF_TOKEN")
if _hf_tok:
    _hf_login(token=_hf_tok)
    print("HF: da dang nhap.")
else:
    print("[CANH BAO] Chua co HF_TOKEN -> model gated (gemma-2-9b-it) se loi 401. "
          "Them secret 'HF_TOKEN' o Colab (icon chia khoa ben trai) roi chay lai cell nay.")

# ============================== CẤU HÌNH ==============================
ROOT = Path('/content/drive/MyDrive/phase2_trackb')   # <-- đổi nếu muốn nơi khác

# --- Chế độ chạy ---
PILOT_MODE    = True     # True = chạy pilot nhỏ. Đặt False SAU KHI qua pilot gate.
PILOT_N_PAIRS = 100      # số cặp cho pilot

# --- Mục tiêu ---
TARGET_PAIRS  = 800      # ha tu 2000 -> vuot tran budget floor 37 units (xem PHASE2_CHECKLIST.md muc 5 Q1)

# --- B4 ensemble ---
N_SELF_CONSISTENCY = 5   # n=5. Cân nhắc hạ 3 sau pilot (câu hỏi #4 của checklist)
TEMPERATURE        = 0.5

# --- Guard-rail B2 ---
LEXICAL_THRESHOLD  = 0.5   # rapidfuzz partial_ratio / 100
NLI_ACCEPT         = {"entailment"}   # neutral + contradiction -> loại (fail-closed)

# --- Debate ---
DEBATE_ROUNDS   = 2      # CỐ ĐỊNH 2. Không lặp đến hội tụ (xem plan §6)
DEBATE_CAP      = 1200

# --- Dịch ---
VI_RATIO = 0.40          # 40% dịch sang VI, 60% giữ EN

# --- Seeds (R-3: ghi lại mọi seed) ---
SEEDS = {
    "pair_sampling":   20260827,
    "debate_roles":    20260828,
    "judge_order":     20260829,
    "generation":      20260830,
    "spotcheck":       20260831,
    "vi_split":        20260832,
}

# --- Models ---
MODELS = {
    "qwen":   {"id": "Qwen/Qwen3-14B-AWQ",           "quantization": "awq",  "dtype": "half"},
    "gemma":  {"id": "google/gemma-2-9b-it",          "quantization": None,   "dtype": "bfloat16"},
    "seallm": {"id": "SeaLLMs/SeaLLMs-v3-7B-Chat",    "quantization": None,   "dtype": "bfloat16"},
    # Judge: GIU Qwen3-14B-AWQ (khong nang cap len 32B) theo quyet dinh ngan sach. Goc lech
    # COMPLEMENTARY o B4/debate KHONG phai do trung checkpoint voi labeler "qwen" -- da xac
    # dinh lai la do chat_prompts() khong tat thinking-mode cua Qwen3 (xem enable_thinking=False
    # trong chat_prompts ben duoi). Sua o do la du, khong can doi judge model.
    "judge":  {"id": "Qwen/Qwen3-14B-AWQ",            "quantization": "awq",  "dtype": "half"},
}
LABELER_KEYS = ["qwen", "gemma", "seallm"]
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
MT_MODEL  = "vinai/vinai-translate-en2vi-v2"

LABELS = ["AGREEMENT","PARTIAL_AGREEMENT","CONTRADICTION",
          "PARTIAL_CONTRADICTION","COMPLEMENTARY","UNRELATED"]

# Trục quan hệ -> xác định nhãn "kề nhau" (plan §5 B5)
AXIS = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY","PARTIAL_CONTRADICTION","CONTRADICTION"]
def is_adjacent(a, b):
    if a == b: return True
    if "UNRELATED" in (a, b):
        other = b if a == "UNRELATED" else a
        return other == "COMPLEMENTARY"        # UNRELATED chỉ kề COMPLEMENTARY
    return abs(AXIS.index(a) - AXIS.index(b)) == 1
# ======================================================================

DIRS = {k: ROOT/k for k in ["raw","interim","processed","reports","configs","prompts"]}
DIRS["transcripts"] = ROOT/"interim"/"debate_transcripts"
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

json.dump(SEEDS, open(DIRS["configs"]/"seeds.json","w"), indent=2)
random.seed(SEEDS["generation"])

print(f"ROOT       = {ROOT}")
print(f"PILOT_MODE = {PILOT_MODE}" + ("   <-- pilot, không phải full run" if PILOT_MODE else "   <-- FULL RUN"))
print(f"n          = {N_SELF_CONSISTENCY}")
print(f"Judge      = {MODELS['judge']['id']}")

## 0.3 · Tiện ích: I/O, checkpoint, budget

`stage()` là context manager làm 3 việc: skip nếu output đã tồn tại, đo thời gian GPU, ghi vào budget log.

In [ ]:
from contextlib import contextmanager

UNITS_PER_HOUR = 11.8          # A100 40GB trên Colab Pro
BUDGET_LOG = DIRS["configs"]/"budget_log.json"
BUDGET_TOTAL = 99.57

def read_jsonl(p):
    p = Path(p)
    if not p.exists(): return []
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

def write_jsonl(p, rows):
    Path(p).write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows)+"\n", encoding='utf-8')
    print(f"  -> {Path(p).name}: {len(rows)} dòng")

def _budget():
    return json.load(open(BUDGET_LOG)) if BUDGET_LOG.exists() else {}

def log_budget(name, hours, note=""):
    b = _budget()
    e = b.get(name, {"hours":0.0,"units":0.0,"runs":0})
    e["hours"] += hours; e["units"] += hours*UNITS_PER_HOUR; e["runs"] += 1; e["note"] = note
    b[name] = e; json.dump(b, open(BUDGET_LOG,"w"), indent=2)

class Skip(Exception): pass

@contextmanager
def stage(name, out_path, force=False):
    # Skip nếu output đã có; đo giờ GPU; ghi budget log.
    out_path = Path(out_path)
    if out_path.exists() and not force:
        n = len(read_jsonl(out_path)) if out_path.suffix=='.jsonl' else '-'
        print(f"[SKIP] {name}: {out_path.name} đã tồn tại ({n} dòng). force=True để chạy lại.")
        yield Skip
        return
    print(f"[RUN ] {name} ...")
    t0 = time.time()
    yield None
    h = (time.time()-t0)/3600
    log_budget(name, h)
    print(f"[DONE] {name}: {h*60:.1f} phút = {h*UNITS_PER_HOUR:.2f} units")

def budget_report():
    b = _budget()
    if not b: print("Chưa có gì được ghi."); return
    print(f"{'Stage':<28}{'Runs':>5}{'Giờ':>9}{'Units':>9}")
    print("-"*51)
    tu = th = 0
    for k,v in b.items():
        print(f"{k:<28}{v['runs']:>5}{v['hours']:>9.2f}{v['units']:>9.2f}")
        tu += v['units']; th += v['hours']
    print("-"*51)
    print(f"{'TỔNG ĐÃ TIÊU':<28}{'':>5}{th:>9.2f}{tu:>9.2f}")
    print(f"{'CÒN LẠI (từ 99.57)':<28}{'':>5}{'':>9}{BUDGET_TOTAL-tu:>9.2f}")
    if BUDGET_TOTAL-tu < 37:
        print("\n[CẢNH BÁO] Dưới sàn an toàn 37 units. Dừng lại, xem checklist mục 'Điều kiện dừng'.")

print("OK. Gọi budget_report() bất cứ lúc nào.")

## 0.4 · Nạp / giải phóng model

Giải phóng VRAM triệt để là bắt buộc — API teardown của vLLM đổi theo version nên bọc try/except; nếu vẫn còn VRAM chiếm sau khi gọi, cách chắc chắn nhất là **Runtime > Restart** rồi chạy tiếp stage kế (mọi thứ đã checkpoint xuống Drive nên không mất gì).

In [ ]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from transformers import AutoTokenizer

_LLM = None; _LLM_KEY = None; _TOK = None

def load_model(key):
    global _LLM, _LLM_KEY, _TOK
    if _LLM_KEY == key:
        return _LLM
    free_model()
    cfg = MODELS[key]
    print(f"Nạp {key}: {cfg['id']} ...")
    kw = dict(model=cfg["id"], dtype=cfg["dtype"], gpu_memory_utilization=0.90,
              max_model_len=4096, trust_remote_code=True, seed=SEEDS["generation"])
    if cfg["quantization"]: kw["quantization"] = cfg["quantization"]
    _LLM = LLM(**kw); _TOK = AutoTokenizer.from_pretrained(cfg["id"], trust_remote_code=True)
    _LLM_KEY = key
    print(f"  OK. VRAM đang dùng: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    return _LLM

def free_model():
    global _LLM, _LLM_KEY, _TOK
    if _LLM is None: return
    print(f"Giải phóng {_LLM_KEY} ...")
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel(); destroy_distributed_environment()
    except Exception as e:
        print(f"  (teardown api: {e})")
    try: del _LLM.llm_engine.model_executor
    except Exception: pass
    del _LLM; _LLM = None; _LLM_KEY = None; _TOK = None
    gc.collect(); torch.cuda.empty_cache()
    print(f"  VRAM còn: {torch.cuda.memory_allocated()/1e9:.1f} GB "
          f"(còn cao -> Runtime > Restart, checkpoint đã an toàn trên Drive)")

def chat_prompts(system, users):
    # Khong phai model nao cung chap nhan role "system" trong chat template
    # (vd. gemma-2-9b-it raise TemplateError: System role not supported) ->
    # fallback: gop system vao dau turn user cho model do.
    #
    # enable_thinking=False: Qwen3 mac dinh bat thinking-mode trong chat template (chen scaffold
    # <think>...</think> truoc cau tra loi that). gen_choice ep grammar chi cho 6 nhan (hoac L1|L2|
    # ABSTAIN o judge) tu TOKEN DAU TIEN, max_tokens=12 -> model khong con cho de "nghi" nhu no
    # duoc huan luyen, roi lai vao dung 1 token an toan/mac dinh bat ke noi dung claim -> day la
    # ly do B4/debate cua rieng Qwen collapse ve COMPLEMENTARY con B2 (gen_json, 320 token, khong
    # ep tu token dau) van on. Tokenizer nao khong nhan tham so nay se tu bo qua (Jinja khong loi
    # voi kwarg khong dung toi) nen an toan de goi chung cho ca 4 model (qwen/gemma/seallm/judge).
    prompts = []
    for u in users:
        try:
            p = _TOK.apply_chat_template(
                    [{"role":"system","content":system},{"role":"user","content":u}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except Exception:
            merged = system + "\n\n" + u
            p = _TOK.apply_chat_template(
                    [{"role":"user","content":merged}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=False)
        prompts.append(p)
    return prompts

def gen_choice(system, users, choices, n=1, temperature=0.0, max_tokens=12):
    # Constrained decoding vào đúng tập choices. Trả list[list[str]] (n kết quả/prompt).
    sp = SamplingParams(temperature=temperature, n=n, max_tokens=max_tokens, seed=SEEDS["generation"],
                        structured_outputs=StructuredOutputsParams(choice=list(choices)))
    outs = _LLM.generate(chat_prompts(system, users), sp)
    return [[o.text.strip() for o in out.outputs] for out in outs]

def gen_json(system, users, schema, temperature=0.2, max_tokens=320):
    sp = SamplingParams(temperature=temperature, n=1, max_tokens=max_tokens, seed=SEEDS["generation"],
                        structured_outputs=StructuredOutputsParams(json=schema))
    outs = _LLM.generate(chat_prompts(system, users), sp)
    res = []
    for out in outs:
        try: res.append(json.loads(out.outputs[0].text))
        except Exception: res.append(None)
    return res

def gen_text(system, users, temperature=0.7, max_tokens=200):
    sp = SamplingParams(temperature=temperature, n=1, max_tokens=max_tokens, seed=SEEDS["generation"])
    return [o.outputs[0].text.strip() for o in _LLM.generate(chat_prompts(system, users), sp)]

print("OK")

## 0.5 · Rubric + few-shot

> **Plan §6:** *"Thứ ăn tiền hơn cả debate là chất lượng rubric. Rubric sắc thì tỉ lệ đồng thuận ở B5 vọt lên và số cặp phải debate giảm hẳn."*

Rubric dưới đây là **bản khởi điểm**. Sau spot-check, chỗ cần sửa chính là các dòng "Ranh giới" — sửa ở đây rồi chạy lại B4.

**Few-shot:** đặt file `fewshot.jsonl` vào `{ROOT}/processed/` để bật. Không có file thì notebook chạy **zero-shot** và in cảnh báo. Format mỗi dòng:
```json
{"left": "...", "right": "...", "label": "PARTIAL_CONTRADICTION", "why": "một câu ngắn"}
```

In [ ]:
RUBRIC = """
Bạn phân loại quan hệ giữa HAI atomic claims của HAI reviewer khác nhau về CÙNG một bài báo.

SÁU NHÃN:
- AGREEMENT: cùng MỘT điểm cụ thể, đồng thuận cả nội dung và mức độ/phạm vi.
- PARTIAL_AGREEMENT: cùng MỘT điểm cụ thể, cùng chiều đánh giá, nhưng khác phạm vi / điều kiện / mức độ.
- CONTRADICTION: cùng MỘT điểm cụ thể, kết luận hoặc lập trường đối lập trực tiếp, không thể cùng đúng.
- PARTIAL_CONTRADICTION: cùng MỘT điểm cụ thể, có mâu thuẫn nhưng bị làm mềm bởi rào đón (hedging) ở một bên.
- COMPLEMENTARY: cùng aspect nhưng KHÁC điểm cụ thể, không mâu thuẫn trực tiếp; một claim bổ sung khía cạnh mà bên kia không nhắc tới.
- UNRELATED: không xác định được điểm chung nào, kể cả khi hai claim được ghép cùng aspect.

CÂU HỎI NEO - hỏi TRƯỚC MỌI THỨ, quyết định cả nhánh còn lại:
Hai claim có đang nói về CÙNG MỘT ĐIỂM CỤ THỂ không (cùng một đoạn / section / thí nghiệm / khẳng định,
KHÔNG chỉ cùng aspect chung chung)? "Thiếu ablation" và "cần thêm baseline" cùng aspect substance nhưng
KHÁC điểm cụ thể -> không phải AGREEMENT/PARTIAL_*/CONTRADICTION*, xét tiếp COMPLEMENTARY/UNRELATED ở mục 3 và 5.
"Thiếu ablation trên dataset Z" và "ablation study chưa đủ" là CÙNG một điểm cụ thể -> xét tiếp
AGREEMENT/PARTIAL_AGREEMENT/PARTIAL_CONTRADICTION/CONTRADICTION theo cùng chiều hay ngược chiều ở mục 1, 2, 4.
COMPLEMENTARY KHÔNG PHẢI nhãn mặc định khi phân vân - nếu không chắc, quay lại CÂU HỎI NEO,
đừng chọn COMPLEMENTARY chỉ vì "nghe có vẻ an toàn".
Tương tự, PARTIAL_AGREEMENT/PARTIAL_CONTRADICTION/COMPLEMENTARY/UNRELATED cũng KHÔNG PHẢI nhãn mặc định an toàn khi ngại chọn AGREEMENT/CONTRADICTION. Diễn giải lại 2 claim về ĐÚNG MỘT khẳng định cốt lõi (cùng một novelty/benefit/mức đóng góp/chất lượng trình bày...) - nếu một bên khẳng định còn bên kia phủ định trực tiếp ĐÚNG khẳng định đó, hoặc cả hai khẳng định khớp nhau, không rào đón -> PHẢI chọn AGREEMENT/CONTRADICTION theo mục 1, 4, 8, 9 bên dưới. Đừng hạ xuống nhãn mềm hơn chỉ vì hai câu dùng từ ngữ bề mặt khác nhau, hoặc vì cảm giác nhãn cực đoan rủi ro hơn.

RANH GIỚI KHÓ - đây là các ranh giới model hay nhầm nhất, đọc kỹ từng dòng:
1. AGREEMENT vs PARTIAL_AGREEMENT (đã xác định là CÙNG một điểm cụ thể, cùng chiều): AGREEMENT = mức độ/phạm vi của hai claim KHỚP nhau, không bên nào thêm điều kiện hay giới hạn mà bên kia không có. PARTIAL_AGREEMENT = cùng chiều nhưng khác mức độ ("thiếu" vs "còn thiếu một chút"), khác phạm vi ("toàn bài" vs "riêng Section 3"), hoặc khác điều kiện đi kèm.
2. COMPLEMENTARY vs PARTIAL_AGREEMENT (dùng CÂU HỎI NEO để phân biệt): khác điểm cụ thể dù cùng aspect -> COMPLEMENTARY. Cùng điểm cụ thể và cùng chiều -> PARTIAL_AGREEMENT hoặc AGREEMENT theo mục 1, KHÔNG PHẢI COMPLEMENTARY dù hai câu dùng từ ngữ khác nhau.
3. COMPLEMENTARY vs PARTIAL_CONTRADICTION (cùng aspect, nhìn qua có vẻ khác chiều): nếu hai claim cùng nói về MỘT điểm cụ thể và một bên có sắc thái phủ định/lo ngại về đúng điểm đó mà bên kia không có -> PARTIAL_CONTRADICTION, không phải COMPLEMENTARY. COMPLEMENTARY chỉ dùng khi hai claim thực sự nói về HAI điểm cụ thể khác nhau, không phải khi một bên "quên nhắc" mặt tiêu cực mà bên kia có nêu.
4. PARTIAL_CONTRADICTION vs CONTRADICTION - trục HEDGING: nếu một bên dùng rào đón ("somewhat", "a bit", "may", "in some cases") làm lập trường KHÔNG còn phủ định toàn bộ bên kia -> PARTIAL_CONTRADICTION. CONTRADICTION chỉ dùng khi hai bên loại trừ nhau, không thể cùng đúng.
5. COMPLEMENTARY vs UNRELATED: còn nhận ra được một điểm/aspect chung, dù đang bàn hai điểm cụ thể khác nhau trong đó -> COMPLEMENTARY. Không xác định được điểm chung nào -> UNRELATED; bị ép vào cùng aspect bucket lúc ghép cặp KHÔNG có nghĩa là chúng liên quan.
6. Stance đối nghịch KHÔNG tự động là CONTRADICTION: một bên khen khía cạnh X, bên kia chê khía cạnh Y của cùng issue (khác điểm cụ thể, xem CÂU HỎI NEO) -> COMPLEMENTARY, không phải CONTRADICTION.
7. Claim tổng quát chung chung ("nice work") ghép với claim rất cụ thể ("thiếu ablation trên dataset Z") -> khác điểm cụ thể (xem CÂU HỎI NEO) -> thường là COMPLEMENTARY, hiếm khi CONTRADICTION.
8. AGREEMENT/PARTIAL_AGREEMENT vs COMPLEMENTARY khi một bên là nhận định TỔNG QUÁT còn bên kia là MỘT VÍ DỤ CỤ THỂ minh hoạ ĐÚNG nhận định đó (không phải một khía cạnh khác): điểm cụ thể ở đây chính là nhận định tổng quát, hai bên cùng chiều -> AGREEMENT/PARTIAL_AGREEMENT theo mục 1, KHÔNG PHẢI COMPLEMENTARY dù B nhắc chi tiết A không có. VD: A "các policy transfer có hiệu suất kém hơn train từ đầu" (nhận định chung) + B "ở Hình 4, policy kém hơn baseline với ResNet-50" (bằng chứng cụ thể cho ĐÚNG nhận định của A) -> AGREEMENT.
9. CONTRADICTION/PARTIAL_CONTRADICTION vs COMPLEMENTARY/UNRELATED khi hai câu dùng từ ngữ HOÀN TOÀN KHÁC NHAU: từ ngữ khác nhau KHÔNG phải bằng chứng cho "khác điểm cụ thể" (xem CÂU HỎI NEO). Diễn giải lại cả 2 câu về MỘT khẳng định cốt lõi (VD "phương pháp X có tính mới hay không", "đóng góp có đủ lớn hay không") - một bên khẳng định, bên kia phủ định trực tiếp ĐÚNG khẳng định đó -> CONTRADICTION/PARTIAL_CONTRADICTION theo mục 4, không phải COMPLEMENTARY/UNRELATED. VD: A [POSITIVE] "Ý tưởng NAS dựa trên importance sampling nghe thú vị" + B [NEGATIVE] "Không có discriminator thì về cơ bản chỉ là REINFORCE với generator phức tạp hơn" - từ ngữ khác hẳn nhau nhưng cùng phán xét tính mới/giá trị của MỘT ý tưởng, hai chiều đối lập -> CONTRADICTION.
"""

def load_fewshot():
    p = DIRS["processed"]/"fewshot.jsonl"
    if not p.exists():
        print("[CẢNH BÁO] Không có fewshot.jsonl -> chạy ZERO-SHOT.")
        print("           3 model sẽ tự hiệu chuẩn theo prior riêng; 'đồng thuận 3/3' ở B5 yếu đi.")
        print("           Xem quyết định treo ở PHASE2_PLAN.md §1.")
        return []
    fs = read_jsonl(p)
    print(f"[OK] Nạp {len(fs)} ví dụ few-shot.")
    from collections import Counter
    print("     Phân bố nhãn:", dict(Counter(f['label'] for f in fs)))
    return fs

FEWSHOT = load_fewshot()

def fewshot_block(labels=None, k=6):
    # labels=None -> TRẢI ĐỀU mọi nhãn bằng round-robin.
    #   Bốc ngẫu nhiên k=6 từ 39 ví dụ chỉ phủ 3/6 nhãn (đã đo) -> prompt B4 thiên
    #   lệch đúng ở chỗ cần cân nhất. Round-robin bảo đảm mỗi nhãn xuất hiện một
    #   lần trước khi bất kỳ nhãn nào lặp lại.
    # labels={L1,L2} -> chỉ ví dụ đúng ranh giới đó, giữ nguyên bốc ngẫu nhiên.
    from collections import defaultdict
    pool = [f for f in FEWSHOT if labels is None or f["label"] in labels]
    if not pool: return ""
    rnd = random.Random(SEEDS["generation"])
    if labels is None:
        by = defaultdict(list)
        for f in pool: by[f["label"]].append(f)
        for v in by.values(): rnd.shuffle(v)
        order = sorted(by)                     # thứ tự nhãn cố định -> tái lập được
        pool = [by[l][i] for i in range(max(map(len, by.values())))
                for l in order if i < len(by[l])]
    else:
        pool = pool[:]; rnd.shuffle(pool)
    out = ["\nVÍ DỤ ĐÃ CÓ NHÃN:"]
    for f in pool[:k]:
        out.append(f'A: "{f["left"]}"\nB: "{f["right"]}"\n-> {f["label"]}'
                   + (f'  ({f["why"]})' if f.get("why") else ""))
    return "\n".join(out) + "\n"

print("\nRubric + few-shot sẵn sàng.")

---
# B1 · Tách câu

Đọc raw → sửa mojibake → **cắt hậu-rebuttal (rule-based, TRƯỚC B2)** → tách câu.

Cắt hậu-rebuttal ở đây, không phải ở B2: nó là so khớp chuỗi thuần túy, không cần LLM. Bỏ qua bước này sẽ ghép claim **đã bị chính người viết rút lại** với claim của reviewer kia → nhãn sai không cách nào phát hiện được ở hạ nguồn.

**Trước khi chạy:** upload file dữ liệu vào `{ROOT}/raw/`.

In [ ]:
# Kiểm tra dữ liệu raw
raw_files = sorted(DIRS["raw"].glob("*.json"))
print(f"Tìm thấy {len(raw_files)} file trong {DIRS['raw']}:")
for f in raw_files: print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
if not raw_files:
    print("\n[DỪNG] Chưa có dữ liệu. Upload file .json (chứa các key Review_*_full) vào thư mục trên.")

In [ ]:
import ftfy
from rapidfuzz import fuzz

REBUTTAL_MARKERS = [
    "UPDATE AFTER", "After reading the author", "post-rebuttal", "=== After rebuttal ==",
    "I have read the author", "-- I have read", "Post revision update", "Post-revision update",
    "After the rebuttal", "Update after rebuttal", "AFTER REBUTTAL",
]

def normalize(t):
    t = ftfy.fix_text(t or "")
    t = t.replace("\u2019","'").replace("\u201c",'"').replace("\u201d",'"').replace("\u2014"," - ")
    t = re.sub(r'\s+\.\s+', '. ', t)
    t = re.sub(r'[ \t]+', ' ', t)
    return t.strip()

def cut_rebuttal(t):
    # Cắt bỏ mọi thứ TỪ marker đầu tiên trở đi. Trả (text, đã_cắt?).
    lo = t.lower(); best = None
    for m in REBUTTAL_MARKERS:
        i = lo.find(m.lower())
        if i != -1 and (best is None or i < best): best = i
    return (t[:best].strip(), True) if best is not None else (t, False)

ABBR = r'(?<!\be\.g)(?<!\bi\.e)(?<!\bet al)(?<!\bFig)(?<!\bEq)(?<!\bcf)(?<!\bvs)(?<!\bDr)(?<!\bMr)(?<!\betc)'
def split_sentences(t):
    t = re.sub(r'\n{2,}', ' <PARA> ', t); t = t.replace('\n', ' ')
    parts = re.split(ABBR + r'(?<=[.!?])\s+(?=[A-Z"\'(\[])', t)
    out = []
    for p in parts:
        for seg in p.split('<PARA>'):
            seg = seg.strip(' -*•\t')
            if 25 <= len(seg) <= 800 and re.search(r'[a-zA-Z]{3}', seg):
                out.append(seg.strip())
    return out

with stage("B1_split", DIRS["interim"]/"sentences.jsonl") as sk:
    if sk is None:
        rows, stats = [], {"papers":0,"review_slots":0,"rebuttal_cut":0,"sentences":0}
        for rf in raw_files:
            data = json.load(open(rf, encoding='utf-8'))
            for pkey, pval in data.items():
                if not isinstance(pval, dict): continue
                paper_id = pkey.split("::")[0]
                stats["papers"] += 1
                # ---- QUAN TRỌNG: trong file IMPACT, "Review_1_full"/"Review_2_full" là VỊ TRÍ,
                # không phải danh tính. Ở entry "...::Review_2_full_vs_Review_4_full", trường tên
                # "Review_1_full" thật ra chứa nội dung của review SỐ 2. Không giải mã đúng thì
                # dedup sẽ gộp nhầm hai review khác nhau thành một.
                real = re.findall(r'Review_(\d+)_full', pkey.split("::")[1]) if "::" in pkey else []
                slots = sorted([k for k in pval if re.fullmatch(r'Review_\d+_full', k)
                                and isinstance(pval[k], str)],
                               key=lambda k: int(re.findall(r'\d+', k)[0]))
                for pos, rkey in enumerate(slots):
                    rid = f"R{real[pos]}" if pos < len(real) else rkey.replace("_full","")
                    stats["review_slots"] += 1
                    txt, cut = cut_rebuttal(normalize(pval[rkey]))
                    if cut: stats["rebuttal_cut"] += 1
                    for i, s in enumerate(split_sentences(txt)):
                        rows.append({"paper_id":paper_id, "review_key":rid,
                                     "sent_id":f"{paper_id}:{rid}:{i:03d}", "text":s})
        # Khử trùng lặp: cùng một review xuất hiện lại ở nhiều pair-entry khác nhau.
        # Giờ sent_id dùng danh tính THẬT nên dedup là đúng, không gộp nhầm.
        seen=set(); ded=[]
        for r in rows:
            if r["sent_id"] in seen: continue
            seen.add(r["sent_id"]); ded.append(r)
        stats["sentences"] = len(ded)
        stats["unique_reviews"] = len({(r["paper_id"], r["review_key"]) for r in ded})
        write_jsonl(DIRS["interim"]/"sentences.jsonl", ded)
        json.dump(stats, open(DIRS["reports"]/"b1_stats.json","w"), indent=2)
        print(f"  {stats}")

sentences = read_jsonl(DIRS["interim"]/"sentences.jsonl")
n_reviews = len({(s['paper_id'], s['review_key']) for s in sentences})   # review DUY NHẤT sau dedup
print(f"\nTổng: {len(sentences)} câu, {len({s['paper_id'] for s in sentences})} paper, {n_reviews} review")
print(f"Trung bình {len(sentences)/max(n_reviews,1):.1f} câu/review   <-- CHECKLIST câu hỏi #2")

---
# B2 · Trích atomic claim + stance + aspect

**Một lượt LLM/câu**, trả 4 trường cùng lúc. Model: **Qwen3-14B** (mạnh nhất) — vì B2 **không có ensemble để tự sửa sai**: lỗi ở đây truyền thẳng xuống B3–B6 mà không bước nào bắt được.

Decoding: **JSON-schema-constrained** (không phải enum thuần như B4) vì trường `claim` là free text.

Lọc 4 nhóm bằng `is_evaluative` (nhóm thứ 5 — hậu-rebuttal — đã cắt ở B1).

In [ ]:
B2_SCHEMA = {
    "type":"object",
    "properties":{
        "is_evaluative":{"type":"boolean"},
        "claim":{"type":"string"},
        "stance":{"enum":["POSITIVE","NEGATIVE","CONCERN","RECOMMENDATION","NEUTRAL"]},
        "aspect":{"enum":["clarity","motivation","substance","soundness",
                          "originality","meaningful comparison","none"]},
    },
    "required":["is_evaluative","claim","stance","aspect"],
}

B2_SYSTEM = """Bạn trích atomic claim từ MỘT câu trong một bài peer review.

is_evaluative = false (và claim = "") nếu câu thuộc một trong bốn nhóm:
  - tóm tắt nội dung bài báo ("This paper proposes...")
  - liệt kê typo / lỗi ngữ pháp / lỗi trình bày ("line 68: ...", "Typos: ...")
  - danh sách tài liệu tham khảo
  - câu hỏi thuần cho tác giả, không kèm phán xét
is_evaluative = true nếu câu mang PHÁN XÉT ĐÁNH GIÁ về bài báo.

claim: viết lại thành MỘT mệnh đề đánh giá độc lập, đứng một mình đọc vẫn hiểu.
  BẮT BUỘC giữ nguyên từ ngữ của câu gốc ở mức tối đa.
  BẮT BUỘC giữ nguyên mọi từ rào đón (somewhat, a bit, may, slightly, arguably...).
  TUYỆT ĐỐI KHÔNG thêm thông tin không có trong câu gốc.
  TUYỆT ĐỐI KHÔNG đảo cực tính (phủ định thành khẳng định hoặc ngược lại).

stance - thứ tự ưu tiên, xét từ trên xuống:
  RECOMMENDATION: có kiến nghị/đề nghị/cần/nên  <- ƯU TIÊN CAO NHẤT.
                  Câu vừa khuyến nghị vừa chê -> RECOMMENDATION, KHÔNG phải CONCERN.
  CONCERN:        nêu lo ngại / hạn chế / thiếu sót
  POSITIVE:       đánh giá tích cực
  NEGATIVE:       phủ định thẳng, không kèm kiến nghị (HIẾM - chỉ dùng khi thực sự khớp)
  NEUTRAL:        còn lại

aspect: clarity | motivation | substance | soundness | originality | meaningful comparison | none
"""

with stage("B2_extract", DIRS["interim"]/"claims_raw.jsonl") as sk:
    if sk is None:
        load_model("qwen")
        batch = sentences[:PILOT_N_PAIRS*6] if PILOT_MODE else sentences
        print(f"  Xử lý {len(batch)} câu" + (" (PILOT)" if PILOT_MODE else ""))
        rows, B = [], 512
        for i in range(0, len(batch), B):
            chunk = batch[i:i+B]
            outs = gen_json(B2_SYSTEM, [c["text"] for c in chunk], B2_SCHEMA)
            for c, o in zip(chunk, outs):
                if not o or not o.get("is_evaluative"): continue
                cl = (o.get("claim") or "").strip()
                if len(cl) < 15: continue
                rows.append({"claim_id":c["sent_id"], "paper_id":c["paper_id"],
                             "review_key":c["review_key"], "source_sentence":c["text"],
                             "claim":cl, "stance":o["stance"], "aspect":o["aspect"]})
            print(f"    {min(i+B,len(batch))}/{len(batch)} câu -> {len(rows)} claim", end="\r")
        print()
        write_jsonl(DIRS["interim"]/"claims_raw.jsonl", rows)

claims_raw = read_jsonl(DIRS["interim"]/"claims_raw.jsonl")
from collections import Counter
print(f"\n{len(claims_raw)} claim thô")
print("stance:", dict(Counter(c['stance'] for c in claims_raw)))
print("aspect:", dict(Counter(c['aspect'] for c in claims_raw)))
print("\n[KIỂM] NEGATIVE phải HIẾM (đường rule production gần như không sinh NEGATIVE).")
print("       NEGATIVE nhiều -> prompt sai thứ tự ưu tiên stance. Xem plan §11 rủi ro #6.")

## B2-guard · Claim verification (2 tầng)

```
claim ──► (1) lexical grounding ──► (2) semantic entailment ──► accept/reject
              rapidfuzz, lọc thô        NLI model, bắt đảo cực tính
```

**Vì sao cần tầng 2:** fuzzy-match chỉ đo trùng lặp **bề mặt từ ngữ**. Nguồn `"does NOT improve robustness"` → claim bịa `"improves robustness"` vẫn đạt điểm cao vì hầu hết từ trùng nhau — mà đây đúng là lỗi tai hại nhất, vì input bị lật cực tính kéo sai toàn bộ nhãn quan hệ ở B4–B6.

`NEUTRAL` → loại (fail-closed, theo triết lý contract). NLI chạy trên GPU nhưng là forward-pass phân loại, chi phí không đáng kể.

In [ ]:
with stage("B2_guard", DIRS["interim"]/"claims.jsonl") as sk:
    if sk is None:
        free_model()      # nhường VRAM cho NLI
        from transformers import pipeline
        nli = pipeline("text-classification", model=NLI_MODEL, device=0, truncation=True)

        kept, rej, t1 = [], [], []
        # --- tầng 1: lexical grounding ---
        for c in claims_raw:
            s = fuzz.partial_ratio(c["claim"].lower(), c["source_sentence"].lower())/100
            c["lex_score"] = round(s,3)
            if s >= LEXICAL_THRESHOLD:
                t1.append(c)
            else:
                rej.append({**c, "reject_reason":"lexical",
                            "detail":f"{s:.2f} < {LEXICAL_THRESHOLD}"})
        print(f"  Tầng 1 (lexical): giữ {len(t1)}/{len(claims_raw)}")

        # --- tầng 2: NLI entailment ---
        B = 64
        for i in range(0, len(t1), B):
            chunk = t1[i:i+B]
            res = nli([{"text":c["source_sentence"], "text_pair":c["claim"]} for c in chunk])
            for c, r in zip(chunk, res):
                lab = r["label"].lower()
                c["nli_label"], c["nli_score"] = lab, round(r["score"],3)
                if lab in NLI_ACCEPT: kept.append(c)
                else: rej.append({**c,"reject_reason":f"nli_{lab}","detail":f"{r['score']:.2f}"})
            print(f"    NLI {min(i+B,len(t1))}/{len(t1)}", end="\r")
        print()

        del nli; gc.collect(); torch.cuda.empty_cache()
        write_jsonl(DIRS["interim"]/"claims.jsonl", kept)
        write_jsonl(DIRS["reports"]/"b2_rejected.jsonl", rej)

        rc = Counter(r["reject_reason"] for r in rej)
        rep = [f"# B2 guard-rail\n", f"- Claim thô: {len(claims_raw)}",
               f"- Giữ lại: {len(kept)} ({len(kept)/max(len(claims_raw),1)*100:.1f}%)",
               f"- Loại: {len(rej)}\n", "| Lý do | Số lượng |","|---|---|"]
        rep += [f"| {k} | {v} |" for k,v in rc.most_common()]
        (DIRS["reports"]/"b2_guard.md").write_text("\n".join(rep), encoding='utf-8')
        print(f"  Loại theo lý do: {dict(rc)}")

claims = read_jsonl(DIRS["interim"]/"claims.jsonl")
kr = len(claims)/max(len(claims_raw),1)*100
print(f"\n{len(claims)} claim qua guard-rail ({kr:.1f}%)   <-- CHECKLIST câu hỏi #3")
if kr < 50:
    print("[CẢNH BÁO] Loại quá nửa. Đọc reports/b2_rejected.jsonl trước khi chạy tiếp —")
    print("           thường là prompt B2 đang cho model diễn giải quá tay, không phải NLI sai.")

---
# B3 · Ghép cặp

Cùng paper + **cùng aspect** + **khác review**. Đúng 2 claim mỗi cặp (R-4: chỉ pairwise).

`pair_id = {paper_id}:{aspect}:{claimA_id}:{claimB_id}`

Vượt mục tiêu → lấy mẫu có seed, **cân bằng theo aspect** để không aspect nào chiếm hết.

In [ ]:
from itertools import combinations
from collections import defaultdict

with stage("B3_pair", DIRS["interim"]/"pairs.jsonl") as sk:
    if sk is None:
        buckets = defaultdict(list)
        for c in claims:
            if c["aspect"] != "none": buckets[(c["paper_id"], c["aspect"])].append(c)

        by_aspect = defaultdict(list)
        for (pid, asp), cs in buckets.items():
            for a, b in combinations(cs, 2):
                if a["review_key"] == b["review_key"]: continue     # phải khác reviewer
                by_aspect[asp].append({
                    "pair_id": f"{pid}:{asp}:{a['claim_id']}:{b['claim_id']}",
                    "paper_id": pid, "aspect": asp,
                    "left":  {"claim_id":a["claim_id"],"text":a["claim"],"stance":a["stance"],
                              "reviewer_alias":"PB-01"},
                    "right": {"claim_id":b["claim_id"],"text":b["claim"],"stance":b["stance"],
                              "reviewer_alias":"PB-02"},
                })
        total = sum(len(v) for v in by_aspect.values())
        print(f"  Sinh được {total} cặp ứng viên: "
              f"{ {k:len(v) for k,v in by_aspect.items()} }")

        target = PILOT_N_PAIRS if PILOT_MODE else TARGET_PAIRS
        rnd = random.Random(SEEDS["pair_sampling"])
        if total > target:
            per = max(1, target // max(len(by_aspect),1))     # cân bằng theo aspect
            pairs = []
            for asp, lst in by_aspect.items():
                rnd.shuffle(lst); pairs += lst[:per]
            leftover = [p for asp,l in by_aspect.items() for p in l[per:]]
            rnd.shuffle(leftover); pairs += leftover[:max(0, target-len(pairs))]
            rnd.shuffle(pairs)
        else:
            pairs = [p for l in by_aspect.values() for p in l]; rnd.shuffle(pairs)

        # R-1: few-shot không bao giờ nằm trong tập train
        fs_txt = {(f["left"].strip(), f["right"].strip()) for f in FEWSHOT}
        if fs_txt:
            before = len(pairs)
            pairs = [p for p in pairs
                     if (p["left"]["text"].strip(), p["right"]["text"].strip()) not in fs_txt
                     and (p["right"]["text"].strip(), p["left"]["text"].strip()) not in fs_txt]
            print(f"  R-1: loại {before-len(pairs)} cặp trùng few-shot")
        write_jsonl(DIRS["interim"]/"pairs.jsonl", pairs)

pairs = read_jsonl(DIRS["interim"]/"pairs.jsonl")
print(f"\n{len(pairs)} cặp -> B4")
print("theo aspect:", dict(Counter(p['aspect'] for p in pairs)))

---
# ⛔ PILOT GATE

**Đang ở `PILOT_MODE = True` thì DỪNG Ở ĐÂY** sau khi chạy hết B4→B6 một lượt trên 100 cặp.

Mở `PHASE2_CHECKLIST.md`, trả lời đủ **5 câu hỏi pilot**, ghi vào bảng ở đó. Chỉ khi đủ 5 câu trả lời mới đặt `PILOT_MODE = False` và chạy full.

Chạy full mà chưa qua gate = đặt cược toàn bộ ngân sách vào ước lượng chưa ai kiểm.

In [ ]:
print("="*62)
print("TRẠNG THÁI PILOT GATE")
print("="*62)
print(f"PILOT_MODE = {PILOT_MODE}")
print(f"Số cặp     = {len(pairs)}")
budget_report()
print("\n5 câu hỏi pilot -> PHASE2_CHECKLIST.md mục 'Pilot gate'.")

---
# B4 · Ensemble gán nhãn

```
3 model × n=5 self-consistency × 2 chiều (A,B) và (B,A) = 30 lượt/cặp
```

**Ba model chạy TUẦN TỰ trên toàn bộ tập** — nạp Qwen, chạy hết mọi cặp, giải phóng, nạp Gemma, ... Không phải 30 lượt liên tiếp cho từng cặp.

**Vì sao n=5 chứ không phải 2:** ở temperature 0.5 mỗi lượt là một mẫu từ phân phối của model. Với n=2, cặp mà model thực sự phân vân có ~50% khả năng ra **hoà 1-1** — phải thêm luật tie-break tuỳ tiện, tái tạo đúng loại nhiễu mà self-consistency sinh ra để khử. Số lẻ luôn có đa số.

Mỗi model ghi checkpoint riêng → đứt kết nối giữa chừng chỉ mất model đang chạy dở.

In [ ]:
B4_SYSTEM = RUBRIC + fewshot_block(k=12) + """
Chỉ trả về ĐÚNG MỘT nhãn trong sáu nhãn trên. Không giải thích.
"""

def b4_prompt(p, flip=False):
    a, b = (p["right"], p["left"]) if flip else (p["left"], p["right"])
    return (f'Claim A [{a["stance"]}]: "{a["text"]}"\n'
            f'Claim B [{b["stance"]}]: "{b["text"]}"\n\nQuan hệ A-B là gì?')

def run_labeler(key):
    out_p = DIRS["interim"]/f"labels_{key}.jsonl"
    with stage(f"B4_{key}", out_p) as sk:
        if sk is not None: return
        load_model(key)
        rows, B = [], 128
        for i in range(0, len(pairs), B):
            chunk = pairs[i:i+B]
            fwd = gen_choice(B4_SYSTEM, [b4_prompt(p,False) for p in chunk], LABELS,
                             n=N_SELF_CONSISTENCY, temperature=TEMPERATURE)
            rev = gen_choice(B4_SYSTEM, [b4_prompt(p,True)  for p in chunk], LABELS,
                             n=N_SELF_CONSISTENCY, temperature=TEMPERATURE)
            for p, f5, r5 in zip(chunk, fwd, rev):
                rows.append({"pair_id":p["pair_id"], "model":key,
                             "samples_fwd":f5, "samples_rev":r5,
                             "vote_fwd":Counter(f5).most_common(1)[0][0],
                             "vote_rev":Counter(r5).most_common(1)[0][0]})
            print(f"    {min(i+B,len(pairs))}/{len(pairs)}", end="\r")
        print()
        write_jsonl(out_p, rows)

for k in LABELER_KEYS:
    run_labeler(k)
free_model()
print("\nXong 3 labeler.")

In [ ]:
# --- Câu hỏi pilot #4: n=3 có đủ thay cho n=5 không? (tính miễn phí trên mẫu đã sinh) ---
diff = tot = 0
for k in LABELER_KEYS:
    for r in read_jsonl(DIRS["interim"]/f"labels_{k}.jsonl"):
        for key in ("samples_fwd","samples_rev"):
            s = r[key]
            if len(s) >= 5:
                tot += 1
                if Counter(s[:3]).most_common(1)[0][0] != Counter(s).most_common(1)[0][0]:
                    diff += 1
pct = diff/max(tot,1)*100
print(f"n=3 khác n=5 ở {diff}/{tot} lượt = {pct:.1f}%   <-- CHECKLIST câu hỏi #4")
print("  < 5%  -> hạ N_SELF_CONSISTENCY = 3 an toàn, tiết kiệm 40% lượt sinh B4")
print("  >= 5% -> tập nhiều ca biên thật, GIỮ n=5")

---
# B5 · Phân luồng

| Tình huống | Xử lý | Weight |
|---|---|---|
| 3/3 đồng thuận **và** cả 3 order-invariant | nhận | 1.0 |
| ≥2/3 đa số **và** order-invariant | nhận | 0.7 |
| Lệch giữa nhãn **kề nhau** | → debate | 0.7 |
| **Lật nhãn khi đảo thứ tự** | → debate | 0.7 |
| Lệch giữa nhãn **xa nhau** | **loại** | — |

Order-invariance phải kiểm **trước** khi đếm phiếu: một cặp trông như "2/3 đồng thuận" nhưng có model lật nhãn thì vẫn phải đi debate — nếu chỉ đếm phiếu thô sẽ bỏ lọt.

In [ ]:
with stage("B5_route", DIRS["interim"]/"routed.jsonl", force=True) as sk:
    if sk is None:
        L = {k:{r["pair_id"]:r for r in read_jsonl(DIRS["interim"]/f"labels_{k}.jsonl")}
             for k in LABELER_KEYS}
        routed = []
        for p in pairs:
            pid = p["pair_id"]
            if not all(pid in L[k] for k in LABELER_KEYS): continue
            votes, obs, flipped, flipped_adjacent = {}, [], [], []
            for k in LABELER_KEYS:
                r = L[k][pid]
                obs += [r["vote_fwd"], r["vote_rev"]]          # 6 quan sát cho soft label
                if r["vote_fwd"] == r["vote_rev"]: votes[k] = r["vote_fwd"]
                else:
                    flipped.append(k)
                    if is_adjacent(r["vote_fwd"], r["vote_rev"]): flipped_adjacent.append(k)
            vl = list(votes.values()); vc = Counter(vl)
            rec = {**p, "votes":votes, "flipped":flipped, "observations":obs}

            if len(vl)==3 and len(vc)==1:
                rec.update(route="unanimous", relation=vl[0], weight=1.0)
            elif flipped_adjacent:
                # lat nhan giua 2 nhan KE NHAU = ca bien that -> debate (nhanh order_flip goc).
                # lat nhan giua 2 nhan XA nhau (vd AGREEMENT<->CONTRADICTION tren cung 1 model) KHONG
                # phai ranh gioi kho -- la model khong on dinh tren rieng cap nay, phieu cua no da bi
                # loai khoi votes/vl o tren roi (chi luu lai o flipped/observations de truy vet), nen
                # roi tiep xuong logic da so/dropped_far ben duoi voi 2 model con lai, khong ep debate
                # dat tien (~50x 1 luot gan nhan) cho mot phieu von da khong dang tin.
                rec.update(route="debate", debate_reason="order_flip")
            elif vc and vc.most_common(1)[0][1] >= 2:
                top = vc.most_common(1)[0][0]
                others = [v for v in vl if v != top]
                if others and not all(is_adjacent(top,o) for o in others):
                    rec.update(route="dropped_far", reason="bất đồng giữa nhãn xa nhau")
                else:
                    rec.update(route="majority", relation=top, weight=0.7)
            else:
                uniq = list(vc)
                if len(uniq)==2 and is_adjacent(*uniq):
                    rec.update(route="debate", debate_reason="adjacent_split")
                else:
                    rec.update(route="dropped_far", reason="bất đồng giữa nhãn xa nhau")

            if rec.get("route")=="debate":
                # hai nhãn tranh chấp = 2 nhãn phổ biến nhất trong 6 quan sát
                cands = [l for l,_ in Counter(obs).most_common(2)]
                if len(cands) < 2:
                    cands += [l for l in LABELS if l not in cands][:2-len(cands)]
                rec["debate_labels"] = cands
            routed.append(rec)
        write_jsonl(DIRS["interim"]/"routed.jsonl", routed)

routed = read_jsonl(DIRS["interim"]/"routed.jsonl")
rc = Counter(r["route"] for r in routed)
print(f"\n{len(routed)} cặp:")
for k,v in rc.most_common(): print(f"  {k:<14} {v:>5}  ({v/len(routed)*100:.1f}%)")
n_deb = rc.get("debate",0)
print(f"\nDebate: {n_deb} cặp ({n_deb/max(len(routed),1)*100:.1f}%) — plan ước ~20%, cap {DEBATE_CAP}")
if n_deb/max(len(routed),1) > 0.35:
    print("[CẢNH BÁO] Tỉ lệ debate cao bất thường -> rubric chưa đủ sắc, hoặc đang chạy zero-shot.")
    print("           Sửa rubric rẻ hơn nhiều so với debate 400+ cặp.")

---
# Debate · 2 vòng cố định

**Đúng 2 vòng, không lặp đến hội tụ.** 5 lượt sinh/cặp = 2 advocate × vòng 1 + 2 advocate × vòng 2 + 1 judge.

Ngân sách cố định, không có API dự phòng → chi phí mỗi cặp debate phải là **hằng số biết trước**, không phụ thuộc việc hai bên có đồng thuận hay không. Qua 2 vòng chưa thuyết phục được Judge thì dùng `ABSTAIN` (loại cặp), không mở vòng 3.

**Bảy biện pháp chống thiên lệch** đều được cài trong code dưới đây: phân vai chỉ định (1), hoán đổi phe theo seed (2), judge không bao giờ là advocate (3), xoá danh tính model (4), ngẫu nhiên thứ tự transcript (5), judge cầm rubric hẹp (6), judge phải trích đoạn quyết định (7).

In [ ]:
debate_pairs = [r for r in routed if r["route"]=="debate"][:DEBATE_CAP]
print(f"{len(debate_pairs)} cặp vào debate")

ADV_SYS = """Bạn là luật sư biện hộ trong một phiên tranh luận về nhãn quan hệ giữa hai claim.
Bạn PHẢI bảo vệ nhãn được giao, kể cả khi bạn không đồng ý.
Tối đa 120 từ. BẮT BUỘC: (a) trích NGUYÊN VĂN đoạn trong claim làm căn cứ;
(b) dẫn điều khoản rubric cụ thể mà bạn dựa vào. Không đề xuất nhãn thứ ba.
"""

JUDGE_SYS = """Bạn là thẩm phán. Hai luật sư đã tranh luận về nhãn quan hệ của một cặp claim.
Bạn KHÔNG tranh luận — bạn phán quyết.
Rubric là mỏ neo: lập luận nghe tự tin không thay thế được rubric.
Đừng mặc định chọn nhãn nghe "an toàn" hơn (PARTIAL_AGREEMENT/PARTIAL_CONTRADICTION/COMPLEMENTARY/UNRELATED) chỉ vì nó đỡ rủi ro hơn AGREEMENT/CONTRADICTION. Nếu rubric và lập luận hai bên cho thấy rõ đây thực sự là AGREEMENT hoặc CONTRADICTION (khẳng định/phủ định trực tiếp cùng một điều, không rào đón), hãy chọn đúng nhãn đó — độ hiếm gặp của một nhãn không phải lý do để né nó.
Trả về đúng một trong ba: nhãn thứ nhất, nhãn thứ hai, hoặc ABSTAIN.
Chọn ABSTAIN khi rubric không phân định được — thà bỏ cặp còn hơn tạo nhãn nhiễu.
"""

def pair_block(r):
    return (f'Claim A [{r["left"]["stance"]}]: "{r["left"]["text"]}"\n'
            f'Claim B [{r["right"]["stance"]}]: "{r["right"]["text"]}"')

def narrow_rubric(l1, l2):
    keep = [ln for ln in RUBRIC.split("\n") if l1 in ln or l2 in ln]
    return (f"RANH GIỚI ĐANG XÉT: {l1} vs {l2}\n" + "\n".join(keep)
            + fewshot_block(labels={l1,l2}, k=4))     # (6)(7)

with stage("debate", DIRS["interim"]/"debate_results.jsonl") as sk:
    if sk is None and debate_pairs:
        rnd = random.Random(SEEDS["debate_roles"])
        # (1)(2) phân vai chỉ định + hoán đổi phe ngẫu nhiên theo từng cặp
        assign = {}
        for r in debate_pairs:
            l1, l2 = r["debate_labels"][:2]
            adv = ["gemma","seallm"]
            if rnd.random() < 0.5: adv = adv[::-1]        # judge (qwen) không bao giờ là advocate (3)
            assign[r["pair_id"]] = {"L1":l1,"L2":l2,"adv_L1":adv[0],"adv_L2":adv[1]}

        openings, rebuttals = defaultdict(dict), defaultdict(dict)

        # ---- VÒNG 1: mở đầu, hai bên viết độc lập, không thấy nhau ----
        for m in ["gemma","seallm"]:
            todo = [(r, "L1" if assign[r["pair_id"]]["adv_L1"]==m else "L2")
                    for r in debate_pairs
                    if m in (assign[r["pair_id"]]["adv_L1"], assign[r["pair_id"]]["adv_L2"])]
            if not todo: continue
            load_model(m)
            prompts = [f'{pair_block(r)}\n\n{narrow_rubric(assign[r["pair_id"]]["L1"], assign[r["pair_id"]]["L2"])}\n\n'
                       f'Bảo vệ nhãn: {assign[r["pair_id"]][side]}' for r, side in todo]
            for (r, side), txt in zip(todo, gen_text(ADV_SYS, prompts, max_tokens=200)):
                openings[r["pair_id"]][side] = txt
            print(f"  vòng 1 - {m}: {len(todo)} bài")

        # ---- VÒNG 2: phản biện, mỗi bên thấy bài mở đầu của bên kia ----
        for m in ["gemma","seallm"]:
            todo = [(r, "L1" if assign[r["pair_id"]]["adv_L1"]==m else "L2")
                    for r in debate_pairs
                    if m in (assign[r["pair_id"]]["adv_L1"], assign[r["pair_id"]]["adv_L2"])]
            if not todo: continue
            load_model(m)
            prompts = []
            for r, side in todo:
                a = assign[r["pair_id"]]; opp = "L2" if side=="L1" else "L1"
                prompts.append(f'{pair_block(r)}\n\n{narrow_rubric(a["L1"], a["L2"])}\n\n'
                               f'Bạn bảo vệ: {a[side]}\n\nBÊN KIA LẬP LUẬN:\n'
                               f'{openings[r["pair_id"]].get(opp,"(không có)")}\n\n'
                               f'Phản bác trực tiếp đoạn trích MẠNH NHẤT của họ.')
            for (r, side), txt in zip(todo, gen_text(ADV_SYS, prompts, max_tokens=200)):
                rebuttals[r["pair_id"]][side] = txt
            print(f"  vòng 2 - {m}: {len(todo)} bài")

        # ---- PHÁN QUYẾT ----
        load_model("judge")
        jr = random.Random(SEEDS["judge_order"]); jprompts, meta = [], []
        for r in debate_pairs:
            a = assign[r["pair_id"]]; pid = r["pair_id"]
            # (4) xoá danh tính model  (5) ngẫu nhiên thứ tự transcript
            t = [("L1", openings[pid].get("L1",""), rebuttals[pid].get("L1","")),
                 ("L2", openings[pid].get("L2",""), rebuttals[pid].get("L2",""))]
            jr.shuffle(t)
            body = "\n\n".join(f"LUẬT SƯ {i+1} (bảo vệ {a[s]}):\nMở đầu: {o}\nPhản biện: {rb}"
                               for i,(s,o,rb) in enumerate(t))
            # (5b) ngau nhien thu tu NHAN trong cau hoi/choice list -- DOC LAP voi thu tu
            # transcript o tren. Khong lam viec nay thi debate_labels[0] (thuong co tan suat
            # raw-vote cao hon, vd COMPLEMENTARY khi labeler "qwen" thien lech, xem B5_route)
            # luon duoc neu "thu nhat" cho MOI cap debate -> position bias he thong cong don
            # tren toan bo tap, khong phai nhieu ngau nhien. jr da dung cho shuffle(t) o tren
            # -> tai lap duoc tu SEEDS["judge_order"].
            q1, q2 = (a["L1"], a["L2"]) if jr.random() < 0.5 else (a["L2"], a["L1"])
            jprompts.append(f'{pair_block(r)}\n\n{narrow_rubric(a["L1"], a["L2"])}\n\n{body}\n\n'
                            f'Phán quyết: {q1}, {q2}, hay ABSTAIN?')
            meta.append((pid, a, t))
        # Gom theo cặp nhãn (L1,L2): constrained decoding chỉ cho phép ĐÚNG L1|L2|ABSTAIN cho
        # từng nhóm. Nếu ép chung cả 6 nhãn, judge có thể trả nhãn thứ ba — plan cấm điều đó.
        groups = defaultdict(list)
        for idx, (pid, a, _t) in enumerate(meta):
            groups[(a["L1"], a["L2"])].append(idx)
        verdict_of = {}
        for (l1, l2), idxs in groups.items():
            res = gen_choice(JUDGE_SYS, [jprompts[i] for i in idxs], [l1, l2, "ABSTAIN"],
                             n=1, temperature=0.0)
            for i, r in zip(idxs, res): verdict_of[i] = r[0]
        rows = []
        for idx, (pid, a, t) in enumerate(meta):
            v = [verdict_of[idx]]
            rows.append({"pair_id":pid, "L1":a["L1"], "L2":a["L2"],
                         "adv_L1":a["adv_L1"], "adv_L2":a["adv_L2"], "verdict":v[0]})
            json.dump({"pair_id":pid, "assignment":a, "transcript_order":[s for s,_,_ in t],
                       "openings":openings[pid], "rebuttals":rebuttals[pid], "verdict":v[0]},
                      open(DIRS["transcripts"]/f"{hashlib.md5(pid.encode()).hexdigest()[:16]}.json","w"),
                      ensure_ascii=False, indent=2)
        write_jsonl(DIRS["interim"]/"debate_results.jsonl", rows)
        free_model()

deb = read_jsonl(DIRS["interim"]/"debate_results.jsonl")
if deb:
    print(f"\n{len(deb)} phán quyết: {dict(Counter(d['verdict'] for d in deb))}")
    ab = sum(1 for d in deb if d["verdict"]=="ABSTAIN")
    print(f"ABSTAIN: {ab} ({ab/len(deb)*100:.1f}%) -> loại khỏi silver set")
    print(f"Transcript lưu tại {DIRS['transcripts']}")

---
# B6 · Nhãn mềm → `trackB_silver.jsonl`

**`soft_label` = phân bố thực nghiệm trên 6 quan sát (3 model × 2 chiều)** — tính đồng nhất cho mọi cặp, không đặc cách. Model lật nhãn tự nhiên đóng góp 0.5/0.5 vào hai nhãn, không cần luật riêng.

**`relation`** = argmax(soft_label) với cặp `unanimous`/`majority`; = **phán quyết Judge** với cặp `debate`.

> ⚠️ **Quyết định cần bạn duyệt:** cặp debate có thể có `relation ≠ argmax(soft_label)`. Đây là chủ đích — nhãn cứng phản ánh **kết quả phân xử**, nhãn mềm phản ánh **độ bất định thô của ensemble**. Train bằng KL trên `soft_label`; `relation` dùng cho eval nhãn cứng và cho việc đọc lại.

Bất đồng 2/3 vs 1/3 **là thông tin**, không phải nhiễu — chính các ca biên này là chỗ model sẽ yếu nhất.

In [ ]:
with stage("B6_soft", DIRS["processed"]/"trackB_silver.jsonl", force=True) as sk:
    if sk is None:
        dmap = {d["pair_id"]:d for d in deb}
        out, skipped = [], Counter()
        for r in routed:
            route = r["route"]
            if route == "dropped_far": skipped["dropped_far"] += 1; continue

            if route == "debate":
                d = dmap.get(r["pair_id"])
                if not d or d["verdict"] == "ABSTAIN":
                    skipped["abstain_or_missing"] += 1; continue
                relation, weight = d["verdict"], 0.7
            else:
                relation, weight = r["relation"], r["weight"]

            obs = r["observations"]                       # 6 quan sát
            soft = {l: round(obs.count(l)/len(obs), 4) for l in LABELS}
            out.append({
                "pair_id": r["pair_id"], "source": "trackB", "lang": "en",
                "left":  {"text": r["left"]["text"],  "stance": r["left"]["stance"]},
                "right": {"text": r["right"]["text"], "stance": r["right"]["stance"]},
                "relation": relation, "soft_label": soft, "weight": weight,
                "provenance": {"paper_id": r["paper_id"], "aspect": r["aspect"],
                               "route": route, "labelers": LABELER_KEYS,
                               "votes": r["votes"], "flipped": r["flipped"]},
            })
        write_jsonl(DIRS["processed"]/"trackB_silver.jsonl", out)
        print(f"  Bỏ: {dict(skipped)}")

silver = read_jsonl(DIRS["processed"]/"trackB_silver.jsonl")
print(f"\n=== trackB_silver.jsonl: {len(silver)} cặp ===")
print("relation:", dict(Counter(s['relation'] for s in silver)))
print("route:   ", dict(Counter(s['provenance']['route'] for s in silver)))
print("weight:  ", dict(Counter(s['weight'] for s in silver)))
miss = [l for l in LABELS if not any(s['relation']==l for s in silver)]
if miss:
    print(f"\n[CẢNH BÁO] Nhãn KHÔNG xuất hiện lần nào: {miss}")
    print("           Model không học được lớp không có mẫu. Xem lại rubric cho các lớp này.")

---
# Spot-check · thay cho bridge đã bỏ

Rút ~50 cặp stratified theo **nhãn** và theo **route** (để ca `debate` — khó nhất — không bị mẫu ngẫu nhiên bỏ sót), xuất CSV để đọc tay.

**Đây không phải κ và không thay thế được bridge:** cỡ mẫu nhỏ, không có gold độc lập, người chấm cũng là người viết rubric. Nó bắt được **lỗi thô và hệ thống**, không chứng minh silver set đúng.

In [ ]:
import csv
N_SPOT = 50
with stage("spotcheck_sample", DIRS["reports"]/"spotcheck_sample.csv", force=True) as sk:
    if sk is None:
        buckets = defaultdict(list)
        for s in silver: buckets[(s["relation"], s["provenance"]["route"])].append(s)
        rnd = random.Random(SEEDS["spotcheck"]); sample = []
        per = max(1, N_SPOT // max(len(buckets),1))
        for b in buckets.values():
            rnd.shuffle(b); sample += b[:per]
        rest = [s for b in buckets.values() for s in b[per:]]
        rnd.shuffle(rest); sample += rest[:max(0, N_SPOT-len(sample))]
        rnd.shuffle(sample)
        with open(DIRS["reports"]/"spotcheck_sample.csv","w",newline='',encoding='utf-8-sig') as f:
            w = csv.writer(f)
            w.writerow(["#","pair_id","aspect","route","claim_A","stance_A","claim_B","stance_B",
                        "nhãn_pipeline","soft_label","BẠN_ĐỒNG_Ý?(y/n)","nhãn_đúng_nếu_n","ghi_chú"])
            for i,s in enumerate(sample,1):
                sl = ", ".join(f"{k}:{v}" for k,v in s["soft_label"].items() if v>0)
                w.writerow([i, s["pair_id"], s["provenance"]["aspect"], s["provenance"]["route"],
                            s["left"]["text"], s["left"]["stance"],
                            s["right"]["text"], s["right"]["stance"],
                            s["relation"], sl, "", "", ""])
        print(f"  {len(sample)} cặp -> spotcheck_sample.csv")

print()
print("Bước tiếp theo (thủ công, không tốn GPU):")
print(f"  1. Tải {DIRS['reports']}/spotcheck_sample.csv")
print("  2. Điền cột 'BẠN_ĐỒNG_Ý?' (y/n) cho từng dòng; nếu n thì ghi nhãn đúng")
print("  3. Ghi kết quả + mẫu lỗi quan sát được vào reports/spotcheck.md")
print("  4. Theo hướng dẫn ở PHASE2_CHECKLIST.md mục 'Spot-check'")
print()
print("Tìm MẪU LỖI HỆ THỐNG, không phải tỉ lệ %:")
print("  - một ranh giới lệch đều một hướng  -> siết ranh giới đó trong RUBRIC, chạy lại B4")
print("  - claim trích hỏng lọt guard-rail   -> chỉnh prompt B2 / ngưỡng NLI")
print("  - một aspect toàn nhãn rác          -> xem lại B3 cho aspect đó")
print("  - route=debate sai nhiều hơn hẳn    -> rubric hẹp đưa Judge chưa đủ sắc")

---
# Dịch sang tiếng Việt

Gán nhãn ở tiếng Anh trước, dịch sau. Giữ **60% EN / dịch 40% VI** — tận dụng cross-lingual transfer của mDeBERTa, giảm nhiễu dịch, tăng robust với code-switch.

**Test giữ nhãn là bắt buộc.** Trục `PARTIAL_*` hoàn toàn là trục hedging; MT chuyên dụng có xu hướng **chuẩn hoá ngôn ngữ rào đón cho gọn**, làm `PARTIAL_CONTRADICTION` trượt thành `CONTRADICTION`.

Ngưỡng leo thang: tỉ lệ lật trên `PARTIAL_*` **> 15%** → chuyển riêng nhóm câu có hedging sang dịch bằng LLM local với prompt giữ tình thái.

In [ ]:
HEDGES = ["may","might","somewhat","a bit","arguably","not necessarily","could","partly",
          "to some extent","rather","slightly","seems","appears"]
def has_hedge(t):
    lo = t.lower(); return any(h in lo for h in HEDGES)

with stage("translate", DIRS["processed"]/"train_vi.jsonl") as sk:
    if sk is None:
        free_model()
        from transformers import AutoTokenizer as AT, AutoModelForSeq2SeqLM
        tk = AT.from_pretrained(MT_MODEL, src_lang="en_XX")
        mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).cuda().eval()

        def tr(texts, B=32):
            out=[]
            for i in range(0,len(texts),B):
                b = tk(texts[i:i+B], return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to("cuda")
                with torch.no_grad():
                    g = mt.generate(**b, decoder_start_token_id=tk.lang_code_to_id["vi_VN"],
                                    num_beams=4, max_length=512)
                out += tk.batch_decode(g, skip_special_tokens=True)
                print(f"    dịch {min(i+B,len(texts))}/{len(texts)}", end="\r")
            return out

        rnd = random.Random(SEEDS["vi_split"])
        idx = list(range(len(silver))); rnd.shuffle(idx)
        vi_idx = set(idx[:int(len(silver)*VI_RATIO)])
        sel = [silver[i] for i in sorted(vi_idx)]
        print(f"  Dịch {len(sel)}/{len(silver)} cặp ({VI_RATIO*100:.0f}%)")
        lt = tr([s["left"]["text"] for s in sel]); rt = tr([s["right"]["text"] for s in sel])
        rows=[]
        for s,a,b in zip(sel,lt,rt):
            r = json.loads(json.dumps(s)); r["lang"]="vi"
            r["left"]["text"], r["right"]["text"] = a, b
            r["pair_id"] = s["pair_id"]+":vi"
            r["provenance"]["translated_from"] = s["pair_id"]
            rows.append(r)
        write_jsonl(DIRS["processed"]/"train_vi.jsonl", rows)
        del mt; gc.collect(); torch.cuda.empty_cache()

train_vi = read_jsonl(DIRS["processed"]/"train_vi.jsonl")
vi_src = {r["provenance"]["translated_from"] for r in train_vi}
train_en = [s for s in silver if s["pair_id"] not in vi_src]
write_jsonl(DIRS["processed"]/"train_en.jsonl", train_en)
print(f"\ntrain_en: {len(train_en)}   train_vi: {len(train_vi)}   tổng: {len(train_en)+len(train_vi)}")

In [ ]:
# --- Test giữ nhãn: chạy labeler trên bản EN và bản VI của CÙNG một mẫu ---
with stage("translation_fidelity", DIRS["reports"]/"translation_fidelity.md") as sk:
    if sk is None and train_vi:
        rnd = random.Random(SEEDS["vi_split"])
        sample = train_vi[:] ; rnd.shuffle(sample); sample = sample[:100]
        by_id = {s["pair_id"]:s for s in silver}
        load_model("qwen")
        en_p = [b4_prompt(by_id[s["provenance"]["translated_from"]]) for s in sample]
        vi_p = [b4_prompt(s) for s in sample]
        en_l = [x[0] for x in gen_choice(B4_SYSTEM, en_p, LABELS, n=1, temperature=0.0)]
        vi_l = [x[0] for x in gen_choice(B4_SYSTEM, vi_p, LABELS, n=1, temperature=0.0)]
        free_model()

        allf = [(e,v) for e,v in zip(en_l,vi_l) if e!=v]
        part = [(s,e,v) for s,e,v in zip(sample,en_l,vi_l) if s["relation"].startswith("PARTIAL")]
        pf   = [1 for _,e,v in part if e!=v]
        r_all = len(allf)/len(sample)*100
        r_par = len(pf)/max(len(part),1)*100
        hedge = sum(1 for s in sample if has_hedge(by_id[s["provenance"]["translated_from"]]["left"]["text"]))

        md_out = [f"# Translation fidelity\n", f"- Mẫu: {len(sample)} cặp",
                  f"- Có hedging: {hedge}", f"- **Lật nhãn tổng: {r_all:.1f}%**",
                  f"- **Lật nhãn nhóm PARTIAL_*: {r_par:.1f}%** (n={len(part)})\n",
                  f"Ngưỡng leo thang: PARTIAL_* > 15%\n",
                  "**VƯỢT NGƯỠNG** -> chuyển nhóm câu hedging sang dịch bằng LLM local (SeaLLM/Qwen3, prompt giữ tình thái)."
                  if r_par > 15 else "Dưới ngưỡng -> giữ VinAI-Translate."]
        (DIRS["reports"]/"translation_fidelity.md").write_text("\n".join(md_out), encoding='utf-8')
        print("\n".join(md_out))

---
# Manifest — Hiếu đang bị chặn

SHA-256 mọi file train/validation → `train_validation_manifest.json`.

Đây là **input bắt buộc Hiếu phải nhận trước khi tạo gold** ([eval-contract §2](../specs/relation-evaluation-contract.md)); field `train_validation_hash_source` của Hiếu chỉ có thể đến từ đây. **Gửi ngay khi có file train đầu tiên**, kể cả bản pilot — đừng đợi chạy xong hết.

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

files = ["train_en.jsonl","train_vi.jsonl","trackB_silver.jsonl","fewshot.jsonl"]
mani = {
    "schema_version":"1.0",
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "pipeline":"track_b_only",
    "pilot_mode": PILOT_MODE,
    "config":{"n_self_consistency":N_SELF_CONSISTENCY,"temperature":TEMPERATURE,
              "labelers":[MODELS[k]["id"] for k in LABELER_KEYS],
              "judge":MODELS["judge"]["id"], "nli_guard":NLI_MODEL, "mt":MT_MODEL,
              "debate_rounds":DEBATE_ROUNDS, "vi_ratio":VI_RATIO},
    "seeds":SEEDS,
    "files":{}, "counts":{},
    "known_limitations":[
        "Không có bridge/κ: dataset chưa từng được đối chiếu với nhãn người trên tập kiểm độc lập.",
        "Toàn bộ nhãn do LLM sinh; không có nhãn người ở bất kỳ đâu trong pipeline.",
        "Kiểm chất lượng duy nhất là spot-check thủ công ~50 cặp, không đủ cỡ mẫu để tính κ.",
        "Domain gap: peer review ML tiếng Anh, khác register với phiếu C4 hội đồng VN.",
        "criterion_id/aspect không nằm trong input model (chỉ trong provenance).",
    ],
}
for fn in files:
    p = DIRS["processed"]/fn
    if p.exists():
        mani["files"][fn] = {"sha256":sha256(p), "bytes":p.stat().st_size}
        mani["counts"][fn] = len(read_jsonl(p))
if not (DIRS["processed"]/"fewshot.jsonl").exists():
    mani["known_limitations"].append("Chạy ZERO-SHOT: không có ví dụ few-shot nào neo hiệu chuẩn 3 model.")

out = ROOT/"train_validation_manifest.json"
json.dump(mani, open(out,"w"), indent=2, ensure_ascii=False)
print(json.dumps(mani, indent=2, ensure_ascii=False))
print(f"\n-> {out}\n\nGỬI FILE NÀY CHO HIẾU NGAY.")

In [ ]:
budget_report()